# DeepFake Detector AI — Training Notebook

This notebook provides a complete pipeline for training the DeepFake Detector AI using the **CIFAKE** dataset on a Google Colab GPU instance.

### **Requirements**
*   Runtime: **GPU** (Runtime > Change runtime type > Hardware accelerator: GPU)
*   Google Drive: Mounted for persistent checkpoint storage.

## 1. Environment Setup

In [ ]:
# @title Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project directory on Drive if it doesn't exist
!mkdir -p /content/drive/MyDrive/deepfake-detector/checkpoints

In [ ]:
# @title Clone Repository & Install Dependencies
import os

# Replace with your GitHub Repo URL
REPO_URL = "YOUR_GITHUB_REPO_URL"
PROJECT_NAME = "deepfake-detector"

%cd /content
if not os.path.exists(PROJECT_NAME):
  !git clone {REPO_URL}

%cd {PROJECT_NAME}
!pip install -r requirements.txt
!pip install torch torchvision --extra-index-url https://download.pytorch.org/whl/cu121  # Ensure CUDA support

## 2. Dataset Preparation
We will download and split the CIFAKE dataset (120k images).

In [ ]:
# @title Run Dataset Setup Script
# This script downloads CIFAKE and creates the train/val/test split automatically
!python scripts/setup_cifake.py

## 3. Training
We will train **EfficientNet-B0** with automated checkpointing to Google Drive.

In [ ]:
# @title Launch Training
from datetime import datetime

RUN_NAME = datetime.now().strftime("%Y%m%d_%H%M%S")
CHECKPOINT_DIR = f"/content/drive/MyDrive/deepfake-detector/checkpoints/{RUN_NAME}"

!python scripts/train.py \
    --train-dir dataset/train \
    --val-dir dataset/val \
    --model-name efficientnet_b0 \
    --batch-size 64 \
    --epochs 30 \
    --checkpoint-dir {CHECKPOINT_DIR} \
    --num-workers 2

## 4. Evaluation
After training, verify the model performance on the test set.

In [ ]:
# @title Run Inference on Test Set
BEST_CKPT = f"{CHECKPOINT_DIR}/best.pth"

!python scripts/evaluate.py \
    --checkpoint {BEST_CKPT} \
    --model-name efficientnet_b0 \
    --test-dir dataset/test